# VM3 — Square: cql 사전학습 → 온라인 td + cql (GitHub에서 바로 여는 판)
`dsrl_colab_calql_axisA_square.ipynb`에서 VM3 역할만 남긴 것. 9번에서 Square npz(`returns`, γ=0.999)를 **이 VM이 만든다**(VM4는 건너뜀). 11번 `METHODS="cql"`, 14번 `METHODS="td cql"`(6 run, 100k). td 사전학습은 VM1이 hq 뒤에 맡는다(`td_square_s*.pt`가 있어야 14번의 td가 뜬다).
순서: 0 → 1(재시작) → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 스모크(step당 0.3 s 넘으면 11번에 `pretrain.cql_n_samples=2` 추가) → 11 → 12 keepalive → (끝나면) 13 → 14 → 15 → 12.

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/dsrl_project'
for d in ['ckpt', 'logs', 'cfg_backup', 'dppo_log']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('project dir:', PROJ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. condacolab (실행 뒤 런타임이 자동 재시작된다. 재시작되면 2번부터)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # 여기서 커널 재시작

## 2. 재시작 후: Drive 다시 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print('ok')

## 3. 저장소 클론 (브랜치 o2o). 두 서브모듈 폴더가 비어 있으면 다음 셀로 다시 받기

In [ ]:
%%bash
git config --global url."https://github.com/".insteadOf "git@github.com:"

cd /content
rm -rf dsrl
git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git
cd dsrl

git log --oneline -1
echo "=== dppo ==="
ls dppo | head -3
echo "=== stable-baselines3 ==="
ls stable-baselines3 | head -3

In [ ]:
%%bash
cd /content/dsrl
git submodule sync --recursive
git submodule update --init --recursive
ls dppo | head

## 4. conda 환경 복원 (Drive 캐시 `env_cache/dsrl_env.tar.gz`, 3~5분)
캐시가 없으면 v3 노트북의 4~5절(설치, 15분)을 대신 돌리고 5b 저장 셀로 캐시를 만들어 둔다.

In [ ]:
%%bash
# 복원: 새 VM에서 4~5번 대신. 1번(condacolab), 2번(Drive), 3번(클론) 뒤에 실행.
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache
mkdir -p /usr/local/envs
cd /usr/local/envs
rm -rf dsrl
tar -xzf $CACHE/dsrl_env.tar.gz
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cap", torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda"); print("matmul ok", (x @ x).sum().item() != 0)
print("robomimic", robomimic.__version__, "robosuite", robosuite.__version__, "mujoco", mujoco.__version__)
PY
echo "restored; continue from section 6"

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. π_dp 체크포인트 (Square). Drive `dppo_log/`에서 config가 기대하는 상대경로로 복사

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p /content/dsrl/dppo/log

if [ -z "$(ls -A $PROJ/dppo_log 2>/dev/null)" ]; then
  echo "--- 첫 다운로드 ---"
  cd /content/dsrl/dppo/log
  gdown --folder https://drive.google.com/drive/folders/1kzC49RRFOE7aTnJh_7OvJ1K5XaDmtuh1
  cp -r /content/dsrl/dppo/log/. $PROJ/dppo_log/
else
  echo "--- Drive에서 복사 ---"
  cp -r $PROJ/dppo_log/. /content/dsrl/dppo/log/
fi
find /content/dsrl/dppo/log -maxdepth 3 | head -40

In [ ]:
%%bash
# Square의 π_dp와 normalization을 config가 기대하는 상대경로에 놓는다 (6번 Can 셀과 같은 방식).
# Drive의 dppo_log 안 어디에 있든 찾아서 복사한다. Square run을 띄울 때만 필요.
set -e
RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log
CKPT_REL=robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
NORM_REL=robomimic/square/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)
echo "checkpoint:    ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"
[[ -n "$CKPT_SRC" && -n "$NORM_SRC" ]] || { echo "square files missing in $DRIVE"; exit 2; }
mkdir -p "$RUNTIME/$(dirname $CKPT_REL)" "$RUNTIME/$(dirname $NORM_REL)"
[[ "$CKPT_SRC" == "$RUNTIME/$CKPT_REL" ]] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[[ "$NORM_SRC" == "$RUNTIME/$NORM_REL" ]] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
echo "=== SQUARE READY ==="; ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"


## 6. 환경 변수

In [ ]:
%%bash
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS

cat /content/env.sh

## 7. 실행 헬퍼 `run_bash`

In [ ]:
# Colab의 %%bash 는 명령이 끝나야 출력을 보여준다. 긴 학습은 이 헬퍼로 돌려서
# 한 줄씩 바로 보이게 하고, 같은 내용을 Drive의 로그 파일에도 남긴다.
import subprocess, sys

NOISE = ("Gym has been unmaintained", "Please upgrade to Gymnasium", "See the migration guide")

def run_bash(script, log_path=None):
    prefix = (
        "source /usr/local/etc/profile.d/conda.sh && conda activate dsrl\n"
        "source /content/env.sh\n"
        "cd /content/dsrl\n"
        "export HYDRA_FULL_ERROR=1 PYTHONUNBUFFERED=1\n"
    )
    log = open(log_path, "a") if log_path else None
    p = subprocess.Popen(["bash", "-c", prefix + script], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if log:
            log.write(line); log.flush()
        if not line.startswith(NOISE):
            print(line, end="", flush=True)
    p.wait()
    if log:
        log.close()
    print(f"\n[exit {p.returncode}]")
    return p.returncode

PROJ = "/content/drive/MyDrive/dsrl_project"
print("run_bash ready")

## 8. site-packages 패치

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /content/dsrl/colab/patch_env.py

## 9. Square 청크 npz 재생성 — `returns`(청크당 γ=**0.999** return-to-go) 추가
원본 hdf5는 9/5에 받아 둔 `robomimic_raw/square/mh/`, 정규화·검증 파일은 Drive `dppo_log/` 아래에서 찾는다(`--check_against`는 공개 train.npz가 있을 때만). 기대: `returns` ≈ [−400, 0].

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
DST=$PROJ/offline/square_train_offline.npz
RAW=$(find $PROJ/robomimic_raw -name "low_dim_v141.hdf5" -path "*square*" | head -n 1)
if [ -z "$RAW" ]; then
  python -m robomimic.scripts.download_datasets --tasks square --dataset_types mh --hdf5_types low_dim --download_dir $PROJ/robomimic_raw
  RAW=$(find $PROJ/robomimic_raw -name "low_dim_v141.hdf5" -path "*square*" | head -n 1)
fi
NORM=$(find $PROJ/dppo_log /content/dsrl/dppo/log -path "*robomimic/square/normalization.npz" | head -n 1)
TRAIN=$(find $PROJ/dppo_log /content/dsrl/dppo/log -path "*robomimic/square/train.npz" | head -n 1)
echo "hdf5: $RAW"; echo "normalization: $NORM"; echo "check_against: ${TRAIN:-none}"
test -n "$RAW" -a -n "$NORM" || { echo "missing inputs"; exit 1; }
CHECK=""; test -n "$TRAIN" && CHECK="--check_against $TRAIN"
cp -f $DST $DST.bak_no_returns 2>/dev/null || true
python scripts/make_offline_chunks.py --load_path "$RAW" --normalization_path "$NORM" $CHECK \
  --gamma 0.999 --n_envs 4 --save_path $DST
python - <<'PY'
import numpy as np
d = np.load("/content/drive/MyDrive/dsrl_project/offline/square_train_offline.npz"); r = d["returns"]
print("keys", sorted(d.files)); print("returns min %.1f  mean %.1f  max %.1f  gamma %s  rows %d" % (r.min(), r.mean(), r.max(), float(d["returns_gamma"]), r.shape[0]))
PY
''')

## 10. 스모크 — td / cql / calql 각 200 step + 증류 100 step. 마지막 줄의 초 × 250 ≈ 본 사전학습 1개 시간(단독 실행 기준)

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
ALPHA=25
D="--config-path=cfg/robomimic --config-name=dsrl_square.yaml offline_data_path=$PROJ/offline/square_train_offline.npz log_dir=$PROJ/logs"
S="pretrain.steps=200 pretrain.distill_steps=100 pretrain.log_every=50 seed=0"
for M in td cql calql; do
  case $M in td) A="pretrain.method=cql pretrain.cql_alpha=0";; cql) A="pretrain.method=cql pretrain.cql_alpha=$ALPHA";; calql) A="pretrain.method=calql pretrain.cql_alpha=$ALPHA";; esac
  T0=$(date +%s)
  python offline_pretrain.py $D $S $A pretrain.out_path=$PROJ/logs/pretrain/smoke_sq_$M.pt 2>&1 \
    | grep "^\[cql\]\|^\[calql\]\|^\[distill\]\|^\[done\]\|Error\|Traceback\|refuse\|ValueError"
  echo "== $M: $(( $(date +%s) - T0 )) s for 200+100 steps (x250 for 50k+25k)"
  rm -f $PROJ/logs/pretrain/smoke_sq_$M.pt $PROJ/logs/pretrain/smoke_sq_${M}_log.csv
done
''')

## 11. 사전학습 본 실행 — `METHODS`를 VM마다 다르게 (예: VM B `cql`, VM C `calql`, VM D `td`), 각 method는 seed 1~3 병렬
결과 `$PROJ/logs/pretrain/{td,cql,calql}_square_s{1,2,3}.pt`, 진행 `$PROJ/logs/pretrain_sq_<method>_s<seed>.out`. 띄운 뒤 12번 keepalive.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
METHODS="cql"
SEEDS="1 2 3"
ALPHA=25
PROJ=/content/drive/MyDrive/dsrl_project; mkdir -p $PROJ/logs/pretrain
D="--config-path=cfg/robomimic --config-name=dsrl_square.yaml offline_data_path=$PROJ/offline/square_train_offline.npz log_dir=$PROJ/logs"
for M in $METHODS; do for SEED in $SEEDS; do
  case $M in
    td)    A="pretrain.method=cql pretrain.cql_alpha=0 pretrain.out_path=$PROJ/logs/pretrain/td_square_s$SEED.pt";;
    cql)   A="pretrain.method=cql pretrain.cql_alpha=$ALPHA";;
    calql) A="pretrain.method=calql pretrain.cql_alpha=$ALPHA";;
  esac
  nohup python offline_pretrain.py $D seed=$SEED $A > $PROJ/logs/pretrain_sq_${M}_s${SEED}.out 2>&1 &
  echo "started $M seed $SEED (pid $!)"
done; done

## 12. keepalive (사전학습·온라인 공용, 자동 반납). 다른 셀을 돌릴 땐 정지 → 실행 → 재실행

In [ ]:
import subprocess, time
PROJ = '/content/drive/MyDrive/dsrl_project'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while True:
    running = sh("ps aux | grep '[o]ffline_pretrain.py\\|[t]rain_dsrl.py' | grep -o 'exp_id=[a-z_0-9]*\\|pretrain.method=[a-z]*\\|seed=[0-9]*' | tr '\\n' ' '")
    ram = sh("free -g | awk 'NR==2{print $3\"/\"$2}'")
    prog = sh("for f in $(ls -t %s/logs/pretrain_sq_*.out %s/logs/square_*.out 2>/dev/null | head -n 12); do "
              "n=$(basename $f .out); l=$(grep '^\\[cql\\]\\|^\\[calql\\]\\|^\\[distill\\]\\|\\[eval\\]\\|\\[done\\]' $f | tail -n 1 | cut -c1-70); "
              "echo -n \"$n: $l | \"; done" % (PROJ, PROJ))
    print(time.strftime('%H:%M'), 'ram', ram, '|', running or '(none running)', '|', prog, flush=True)
    if not running:
        print('all done -> unassigning runtime', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)

## 13. 사전학습 확인 (읽는 법은 Can 노트북 13번과 같음: cql의 `q_ood_mean` 폭주 = 눈금 붕괴 → ALPHA 낮춤; calql `floored_frac` 0.2~0.8)

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
ls -lh $PROJ/logs/pretrain/{td,cql,calql}_square_s*.pt 2>/dev/null
for F in $PROJ/logs/pretrain_sq_*.out; do echo "== $(basename $F): $(grep '^\[done\]\|Error\|Traceback' $F | tail -n 1)"; done
for M in td cql calql; do for S in 1 2 3; do
  F=$PROJ/logs/pretrain/${M}_square_s${S}_log.csv; test -f $F && echo "-- $M s$S: $(grep -v distill $F | tail -n 1 | cut -d, -f2,3,5,9,11,12,13)"
done; done
echo "(columns: phase, step, critic_loss, q_mean, penalty, q_ood_mean, floored_frac)"

## 14. 온라인 — `METHODS` × seed 1~3, 기본 100k (시간 여유가 있으면 `STEPS=150000`)
축 A 통제 그대로. 두 VM으로 나누려면 METHODS를 나눈다(예: 한쪽 `td cql`, 다른쪽 `calql`). 5 run/VM이면 run당 속도가 9 run/VM보다 빠르다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
METHODS="td cql"
SEEDS="1 2 3"
STEPS=100000
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_square.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=$STEPS offline_mix.mode=none load_offline_data=False"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for M in $METHODS; do for S in $SEEDS; do
  case $M in td) V=cql;; *) V=$M;; esac
  launch square_${M}_s$S seed=$S variant=$V pretrain_path=$PROJ/logs/pretrain/${M}_square_s$S.pt
done; done

## 15. 온라인 확인 (3~5분 뒤). 기대: `[pretrain] cql|calql: loaded critic, critic_target, critic_noise`(actor 없음), `[eval] env_steps=0`; 첫 학습 평가는 37k(약 1시간 뒤)

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
echo "processes: $(ps aux | grep -c '[t]rain_dsrl.py')"
for F in $(ls -t $PROJ/logs/square_{td,cql,calql}_s*.out 2>/dev/null); do
  echo "== $(basename $F .out): $(grep '\[pretrain\]\|\[eval\]\|Error\|Traceback' $F | tail -n 1 | cut -c1-110)"
done
free -g | head -2

## 16. (선택) `calql_t12i` — Square에서는 목표 엔트로피 12가 후반을 잃었으므로(HANDOFF 19.8) 우선순위 낮음. 필요하면 Can 노트북 16번과 같은 방식

## 17. 결과 zip (CPU 런타임에서도 됨). 로컬: `plot_results.py --axes "square_critic=square_baseline,square_td,square_iql,square_cql,square_calql"`

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip